# CAS Exam 5: Berquist-Sherman Adjustments

This notebook uses sample data from:
- `.../chainladder-python/chainladder/utils/data/berqsherm.csv`

Learning goals:
- Explain why adjustments are data transformers (not development estimators).
- Cover all Berquist-Sherman adjustment points and formulas shown in your screenshots.
- Demonstrate practical adjustment scenarios in `chainladder` before running development methods.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT / 'chainladder-python') not in sys.path:
    sys.path.insert(0, str(ROOT / 'chainladder-python'))

import chainladder as cl

triangle = cl.load_sample('berqsherm').loc['MedMal']
triangle


,Triangle Summary
Valuation:,1976-12
Grain:,OYDY
Shape:,"(1, 4, 8, 8)"
Index:,[LOB]
Columns:,"[Incurred, Paid, Reported, Closed]"


## Step 0: Diagnose Before You Adjust

Berquist-Sherman adjustments are only appropriate when diagnostics confirm a structural distortion in the triangle. Running B-S without diagnostic evidence introduces unnecessary judgment and can create artificial smoothness. The three diagnostics below are the standard pre-adjustment screen.

In [ ]:
from reservingengine.reserving import (
    paid_vs_incurred_comparison,
    case_adequacy_indicator,
    calendar_year_diagnostic,
)

paid_tri = triangle['Paid']
incurred_tri = triangle['Incurred']

pi = paid_vs_incurred_comparison(paid_tri, incurred_tri)
print("LDF comparison — paid vs incurred (ldf_ratio = paid_ldf / incurred_ldf):")
pi['ldf_comparison'].style.format("{:.4f}")

**Reading the LDF comparison:**

- `ldf_ratio > 1` at early development ages (e.g., 12-36 months): paid claims are developing FASTER than incurred — paid is catching up to incurred faster than expected. This is the signal for a **settlement rate speedup**, which is what the B-S paid adjustment corrects.
- `ldf_ratio ≈ 1` across all ages: paid and incurred are developing at the same rate — no evidence of settlement rate distortion. B-S paid adjustment is unlikely to be material.
- `ldf_ratio < 1` at early ages: normal and expected — paid claims naturally lag incurred at early development ages.

**EXAM RED FLAG —** A ldf_ratio materially above 1.0 at 12-36 months (with the effect diminishing at later ages) is the textbook signal for Berquist-Sherman paid adjustment.

In [ ]:
print("Latest-diagonal paid-to-incurred by accident year:")
pi['paid_to_incurred_latest'].style.format({
    'latest_paid': '{:,.0f}',
    'latest_incurred': '{:,.0f}',
    'paid_to_incurred': '{:.4f}',
})

In [ ]:
ca = case_adequacy_indicator(paid_tri, incurred_tri)
print(f"Case adequacy trend slope: {ca['trend_slope'].iloc[0]:.6f}")
print()
print("Paid/incurred by accident year:")
ca.style.format({'paid_to_incurred': '{:.4f}', 'trend_slope': '{:.6f}'})

**Reading the case adequacy indicator:**

- **Negative trend slope**: paid/incurred ratio is declining across origin years — case reserves are becoming more adequate over time (case reserves grow faster than paid). This is the signal for **B-S reported (incurred) adjustment**.
- **Positive trend slope**: paid/incurred ratio is rising — paid claims growing relative to incurred, or case adequacy weakening.
- **Near-zero slope**: stable case adequacy — no B-S reported adjustment needed.

**EXAM RED FLAG —** A negative slope combined with the most recent origin years showing paid/incurred well below 1.0 is the strongest indicator for Berquist-Sherman reported adjustment. The adjustment restates historical incurred claims as if case reserves had always reflected the current (latest-diagonal) adequacy level.

In [ ]:
cy = calendar_year_diagnostic(paid_tri)
print("L/S matrix (read diagonals for calendar year patterns):")
display(cy['ls_matrix'])
print("Calendar year summary (pct_L ≥ 0.75 = above-average development year):")
cy['calendar_year_summary']

**L/S test connection to Berquist-Sherman:**

- **Calendar year effects detected (pct_L ≥ 0.75) AND paid LDF > incurred LDF**: Settlement rate speedup — B-S paid adjustment indicated.
- **Calendar year effects detected AND paid/incurred declining across origin years**: Case adequacy change — B-S reported adjustment indicated.
- **Calendar year effects detected BUT paid/incurred comparisons are clean**: The distortion is likely from **external factors** (e.g., economic inflation, legal environment shifts) rather than internal operational changes. B-S is NOT the right tool — consider a trend adjustment instead.

**EXAM RED FLAG —** Do not apply Berquist-Sherman solely because the L/S test flags a calendar year. The L/S test identifies that something is wrong in that diagonal; the `paid_vs_incurred_comparison` and `case_adequacy_indicator` identify WHETHER it is a B-S-addressable distortion.

See `exam5_diagnostics.ipynb` Sections 3–5 for full diagnostic walkthroughs.

## Formula Sheet Reference

### When Berquist-Sherman Is Needed

Run diagnostics BEFORE applying adjustments. The three key diagnostics are:

| Signal | Diagnostic | Indication |
|---|---|---|
| Paid develops faster than incurred at early development ages | `paid_vs_incurred_comparison`: ldf_ratio > 1 at early ages | B-S paid adjustment (settlement rate speedup) |
| Paid/incurred ratios declining across origin years | `case_adequacy_indicator`: negative trend_slope | B-S reported adjustment (case adequacy strengthening) |
| Calendar year diagonal effects detected | `calendar_year_diagnostic`: pct_L ≥ 0.75 | Investigate — may be B-S territory or external inflation |

### B-S Paid Claim Adjustment (Settlement Rate Change)

If historical disposal rate < selected disposal rate (settlement speedup: historical paid is too low):

$$\text{AdjPaid}_i = \text{Paid}_i + (\text{Paid}_{i+1} - \text{Paid}_i) \times \frac{\text{DR}_{sel,i} - \text{DR}_{hist,i}}{\text{DR}_{hist,i+1} - \text{DR}_{hist,i}}$$

If historical disposal rate > selected disposal rate (settlement slowdown: historical paid is too high):

$$\text{AdjPaid}_i = \text{Paid}_i - (\text{Paid}_i - \text{Paid}_{i-1}) \times \frac{\text{DR}_{hist,i} - \text{DR}_{sel,i}}{\text{DR}_{hist,i} - \text{DR}_{hist,i-1}}$$

Where DR = Disposal Rate = Cumulative Closed Counts / Ultimate Reported Counts.

### B-S Reported Claim Adjustment (Case Outstanding Adequacy Change)

$$\text{Average Case OS}_{w,d} = \frac{\text{Incurred}_{w,d} - \text{Paid}_{w,d}}{\text{Reported Count}_{w,d} - \text{Closed Count}_{w,d}}$$

Trend the latest diagonal average case OS backward to a common level:

$$\text{Adj. Avg Case OS}_{w,d} = \text{Latest Avg Case OS}_w \times (1 - \text{trend})^{(\text{latest age} - d)/12}$$

$$\text{Adj. Incurred}_{w,d} = \text{Adj. Avg Case OS}_{w,d} \times (\text{Reported Count}_{w,d} - \text{Closed Count}_{w,d}) + \text{Paid}_{w,d}$$

### Combined B-S Adjustment (Both Distortions Present)

$$\text{Adj. Incurred}_{w,d} = \text{Adj. Avg Case OS}_{w,d} \times (\text{Reported Count}_{w,d} - \text{Adj. Closed Count}_{w,d}) + \text{Adj. Paid}_{w,d}$$

**Key principle:** Adjustments modify the INNER triangle cells while preserving the latest diagonal. The latest-diagonal values represent the most current information and are assumed correct.

### Workflow Placement

```
Raw Triangle → Berquist-Sherman Adjustment → Development Estimation → Tail → Reserving Method
```

Adjustments MUST precede factor selection; otherwise, LDFs absorb the distortion and the adjustment is redundant.

### a. Purpose

Applied when:
- operational changes distort development patterns
- case reserving practices change
- payment speed shifts
- structural breaks exist

Goal: restore comparability across origin years before estimating LDFs.

### b. Berquist-Sherman

Corrects for:
- changes in case reserve adequacy
- changes in claim settlement rates

Mechanics (conceptually):
- adjust reported losses or case reserves to reflect a consistent reserving standard
- adjust paid amounts to reflect consistent closure speed

Often requires:
- case outstanding triangle
- paid triangle
- reported counts (for settlement rate)

Result: adjusted triangle better aligned with steady-state assumptions.
Used prior to development and Chain Ladder.

### c. Trend Adjustments

Applies inflation or trend to historical data to bring values to a common cost level.

Conceptually:
- multiply origin/development cells by trend factors
- typically applied before development estimation if cost levels vary materially

Important distinction:
- trending is a structural adjustment, not a development selection

### d. On-Level / Exposure Adjustments

Adjust for:
- rate level changes
- exposure distortions
- structural shifts in business mix

Often applied when triangle data must be aligned with a consistent pricing/exposure base.

### e. Workflow Placement

Typical order:
Raw Triangle
-> Adjustment (e.g., Berquist-Sherman, trend)
-> Development estimation
-> Tail
-> Reserving method

Adjustments should precede factor selection; otherwise LDFs absorb distortion.

### f. When Adjustments Are Necessary

Indicators:
- calendar-year effects detected by valuation correlation tests
- visible kinks or breaks in development diagonals
- operational changes known from business context
- abnormal settlement or case reserve patterns

If distortions exist and are uncorrected, LDFs will be biased.

### g. Conceptual Distinction

Development assumes:
- homogeneity across origin years

Adjustments aim to restore homogeneity before estimating development.
They are data corrections, not modeling choices.

### h. Practical Notes

- Adjustments introduce judgment; document assumptions.
- Always compare unadjusted results and adjusted results.
- Over-adjustment can introduce artificial smoothness.

Core idea: adjustments correct structural distortions in the triangle so that development patterns reflect true claim emergence rather than operational artifacts.


## Step 1: Berquist-Sherman Data Requirements and Scope

`chainladder.BerquistSherman` is a data adjustment transformer.

Required triangle columns:
- `paid_amount`
- `incurred_amount`
- `reported_count`
- `closed_count`

In the standard implementation, the adjustment pipeline restates inner diagonals while preserving the latest diagonal.
Reported claim counts are used to derive disposal rates and support settlement-rate normalization.


In [ ]:
required_columns = ['Paid', 'Incurred', 'Reported', 'Closed']
print('Triangle columns:', [str(c) for c in triangle.columns])

triangle[required_columns].to_frame(origin_as_datetime=False).head(12)


## Step 2: Berquist-Sherman Paid Claim Development Adjustment

Focus: settlement rate adjustment.

Key points:
- Restate paid claims by interpolating paid amounts between historical disposal rates and selected (usually most recent) disposal rates by maturity.
- Assumes disposal rates are roughly proportional to the percentage of ultimate claims paid at each maturity.

Mechanics:
1. Analyze historical disposal rates. If disposal rates changed materially, a paid adjustment is warranted.
2. Restate cumulative paid claims.
   - Select disposal rates by maturity.
   - Interpolate between paid claims of adjacent maturities.
   - If historical disposal rate is below selected disposal rate, adjust paid upward.
   - If historical disposal rate is above selected disposal rate, adjust paid downward.
3. Perform paid development on adjusted paid data.

Paid B-S interpolation formulas:

If historical disposal rate < selected disposal rate:

`AdjPaid_i = Paid_i + (Paid_{i+1} - Paid_i) * ((DR_sel,i - DR_hist,i) / (DR_hist,i+1 - DR_hist,i))`

If historical disposal rate > selected disposal rate:

`AdjPaid_i = Paid_i - (Paid_i - Paid_{i-1}) * ((DR_hist,i - DR_sel,i) / (DR_hist,i - DR_hist,i-1))`

Alternative two-point exponential fit (used in some exam setups):
- Compute adjusted cumulative closed claim counts by applying selected disposal rates to selected ultimate reported counts.
- Fit/use `Y = a * e^(bX)` where `X` is adjusted closed count and `Y` is adjusted paid amount proxy.
- If historical claim count is below adjusted count, paid is adjusted upward; if above, paid is adjusted downward.


In [ ]:
reported_ult = cl.Chainladder().fit(triangle['Reported']).ultimate_
disposal_rate = triangle['Closed'] / reported_ult

print('Disposal-rate triangle (Closed / Ultimate Reported):')
disposal_rate.to_frame(origin_as_datetime=False).head(12)


## Step 3: Berquist-Sherman Reported Claim Development Adjustment

Focus: case outstanding adequacy adjustment.

Key points:
- Restate reported claims to a common level of case outstanding adequacy.
- Assumes changes in average case outstanding by maturity reflect adequacy changes and/or severity trend.

Mechanics:
1. Evaluate data:
   - Analyze percent change in claim severity.
   - Analyze percent change in average case outstanding.
   - If these diverge materially, a reported adjustment may be warranted.
2. Restate cumulative reported claims:
   - Select a severity trend (if given by the exam question, use it directly).
   - Trend latest average case outstanding backward.
   - Reconstruct reported claims using adjusted average case outstanding.
3. Perform reported development on adjusted reported data.

Reported B-S formulas:

`AverageCaseOS = (ReportedClaims - PaidClaims) / (ReportedCounts - ClosedCounts)`

`AdjReportedClaims = AdjAverageCaseOS * (ReportedCounts - ClosedCounts) + PaidClaims`


In [ ]:
impact_table = pd.DataFrame(
    [
        [
            'Settlement rate speedup (paid claims closing faster)',
            'Historical paid LDFs are biased upward at early ages — future development is overstated',
            'B-S paid adjustment restates historical paid to a consistent disposal-rate basis; reduces upward bias in early LDFs',
        ],
        [
            'Increase in case outstanding adequacy (strengthening reserves)',
            'Historical incurred LDFs are biased upward — the incurred triangle reflects progressively stronger reserves, making older development look smaller than it actually was',
            'B-S reported adjustment restates historical incurred to a consistent case OS adequacy basis; reduces upward bias in incurred LDFs',
        ],
        [
            'Both distortions present simultaneously',
            'Both paid and incurred triangles are distorted; unadjusted LDFs from either basis are biased',
            'Combined B-S: adjust paid first (disposal rate normalization), then restate incurred using adjusted paid and adjusted closed counts',
        ],
        [
            'No distortion detected (diagnostics clean)',
            'Neither LDF pattern is biased by operational changes',
            'B-S adjustment is NOT warranted — applying it without evidence introduces artificial smoothness and unjustified judgment',
        ],
        [
            'External inflation (no operational change)',
            'Calendar year effects are present but both paid and incurred triangles show the effect equally',
            'B-S does NOT address external inflation — use a separate trend adjustment; applying B-S in this scenario is inappropriate',
        ],
    ],
    columns=['Change in Environment / Situation', 'Effect on Triangle', 'Berquist-Sherman Response'],
)

impact_table

## When NOT to Use Berquist-Sherman

**EXAM RED FLAG —** B-S is NOT appropriate when:
1. **No diagnostic evidence of distortion**: Do not adjust without evidence from `paid_vs_incurred_comparison` or `case_adequacy_indicator`.
2. **Fewer than 5–6 origin years with inner-triangle data**: Too few years to calibrate the adjustment — the trend parameter and disposal rate selections will have very high uncertainty.
3. **Externally-driven distortion**: If the calendar year effects are caused by external factors (inflation, legal environment) rather than internal operational changes, B-S does not address the root cause.
4. **Over-adjustment risk**: B-S can make the adjusted triangle appear artificially smooth — especially if the trend parameter is set too high. Always compare unadjusted and adjusted results and document the judgment.

In [ ]:
open_counts = triangle['Reported'] - triangle['Closed']
avg_case_os = (triangle['Incurred'] - triangle['Paid']) / open_counts

latest_avg_case_os = avg_case_os.latest_diagonal.to_frame().iloc[:, 0]
latest_paid = triangle['Paid'].latest_diagonal.to_frame().iloc[:, 0]
latest_reported = triangle['Reported'].latest_diagonal.to_frame().iloc[:, 0]

diagnostic = pd.DataFrame(
    {
        'Latest Avg Case OS': latest_avg_case_os,
        'Latest Paid': latest_paid,
        'Latest Reported': latest_reported,
    }
)

diagnostic['Avg Case OS YoY %'] = diagnostic['Latest Avg Case OS'].pct_change()
diagnostic['Paid YoY %'] = diagnostic['Latest Paid'].pct_change()
diagnostic['Reported YoY %'] = diagnostic['Latest Reported'].pct_change()

diagnostic


## Step 4: Combined Change in Settlement Rate and Case Outstanding Adequacy

When both distortions are present:
1. Perform the B-S paid adjustment to obtain adjusted cumulative paid claims.
2. Perform the B-S reported adjustment to obtain adjusted average case outstanding (not final adjusted reported yet).
3. Adjust closed claim counts by selected disposal rates.
4. Restate adjusted reported claims using adjusted average case outstanding, adjusted open counts, and adjusted paid claims.

Combined formula:

`AdjReportedClaims = AdjAverageCaseOS * (ReportedCounts - AdjClosedCounts) + AdjPaidClaims`


In [ ]:
berq = cl.BerquistSherman(
    paid_amount='Paid',
    incurred_amount='Incurred',
    reported_count='Reported',
    closed_count='Closed',
    trend=0.15,
).fit(triangle)

adjusted = berq.adjusted_triangle_

reported_ratio = (triangle / adjusted)['Reported'].to_frame(origin_as_datetime=False)
paid_ratio = (triangle / adjusted)['Paid'].to_frame(origin_as_datetime=False)
incurred_ratio = (triangle / adjusted)['Incurred'].to_frame(origin_as_datetime=False)

print('Reported ratio (original / adjusted): expected ~1.0 where observed')
display(reported_ratio)

print('Paid ratio (original / adjusted):')
display(paid_ratio)

print('Incurred ratio (original / adjusted):')
display(incurred_ratio)


## Step 5: All Practical Berquist-Sherman Adjustment Levers in `chainladder`

Available adjustment controls:
1. `trend`: controls the case-outstanding adequacy normalization.
2. `reported_count_estimator`: controls the estimator used to derive ultimate reported counts and disposal rates.
3. `transform`: outputs an adjusted triangle to feed into Development/Chain Ladder/Tail workflows.

Below, we run sensitivity across trends plus an alternative reported-count estimator.


In [ ]:
def inner_abs_change(original: cl.Triangle, adjusted_tri: cl.Triangle, column: str) -> float:
    original_df = original[column].to_frame(origin_as_datetime=False)
    adjusted_df = adjusted_tri[column].to_frame(origin_as_datetime=False)

    total = 0.0
    for origin in original_df.index:
        row = original_df.loc[origin]
        observed_ages = row.dropna().index.tolist()
        if len(observed_ages) <= 1:
            continue
        for age in observed_ages[:-1]:
            diff = adjusted_df.loc[origin, age] - original_df.loc[origin, age]
            if np.isfinite(diff):
                total += abs(float(diff))
    return total

alt_reported_estimator = cl.Pipeline(
    steps=[
        ('dev', cl.Development(average='simple')),
        ('model', cl.Chainladder()),
    ]
)

scenario_models = {
    'trend_0_default_estimator': cl.BerquistSherman(
        paid_amount='Paid', incurred_amount='Incurred', reported_count='Reported', closed_count='Closed', trend=0.00
    ),
    'trend_15_default_estimator': cl.BerquistSherman(
        paid_amount='Paid', incurred_amount='Incurred', reported_count='Reported', closed_count='Closed', trend=0.15
    ),
    'trend_25_default_estimator': cl.BerquistSherman(
        paid_amount='Paid', incurred_amount='Incurred', reported_count='Reported', closed_count='Closed', trend=0.25
    ),
    'trend_15_alt_reported_estimator': cl.BerquistSherman(
        paid_amount='Paid',
        incurred_amount='Incurred',
        reported_count='Reported',
        closed_count='Closed',
        trend=0.15,
        reported_count_estimator=alt_reported_estimator,
    ),
}

rows = []
for name, model in scenario_models.items():
    fit_model = model.fit(triangle)
    adj_tri = fit_model.adjusted_triangle_

    latest_paid_same = np.allclose(
        triangle['Paid'].latest_diagonal.values,
        adj_tri['Paid'].latest_diagonal.values,
        equal_nan=True,
    )

    rows.append(
        {
            'Scenario': name,
            'InnerAbsChange_Paid': inner_abs_change(triangle, adj_tri, 'Paid'),
            'InnerAbsChange_Incurred': inner_abs_change(triangle, adj_tri, 'Incurred'),
            'InnerAbsChange_Closed': inner_abs_change(triangle, adj_tri, 'Closed'),
            'LatestDiag_PaidUnchanged': bool(latest_paid_same),
        }
    )

pd.DataFrame(rows).set_index('Scenario').sort_index()


## Step 6: Workflow Placement and Interpretation

Use adjusted data in the full reserving workflow:
- Raw Triangle
- Berquist-Sherman (and/or trend, on-level)
- Development factor selection
- Tail selection
- Reserving method

Why this matters:
- If you skip adjustment when distortion exists, LDFs absorb operational artifacts.
- Adjustments restore comparability; development then estimates signal, not noise.


In [ ]:
raw_cl_paid = cl.Chainladder().fit(triangle['Paid'])
adj_cl_paid = cl.Chainladder().fit(adjusted['Paid'])

raw_ultimate = raw_cl_paid.ultimate_.to_frame().iloc[:, 0]
adj_ultimate = adj_cl_paid.ultimate_.to_frame().iloc[:, 0]

ultimate_compare = pd.DataFrame(
    {
        'Raw Ultimate (Paid CL)': raw_ultimate,
        'Adjusted Ultimate (Paid CL)': adj_ultimate,
    }
)
ultimate_compare['Difference'] = (
    ultimate_compare['Adjusted Ultimate (Paid CL)'] - ultimate_compare['Raw Ultimate (Paid CL)']
)

ultimate_compare


## Practical Exam Checklist

- State the distortion: settlement rate change, case adequacy change, or both.
- State why adjustment is needed (diagnostic evidence + business context).
- Show selected assumptions (trend, disposal-rate basis, estimator choice).
- Restate the triangle first, then estimate development and reserve.
- Show both unadjusted and adjusted indications.
- Warn against over-adjustment and document judgment clearly.
